# Lecture 20: Attention, Transformers & Large Language Models

In L19, we saw that word embeddings capture meaning as vectors, but each word gets one fixed vector regardless of context. Today, we'll see how **attention** solves this, and then use a real language model (SmolLM2) to generate text and answer questions.

## Setup

We'll use the `transformers` library from Hugging Face to work with **SmolLM2-135M-Instruct** --- a 135-million-parameter open-weight model from Hugging Face (2024). It is small enough to load in a few seconds on Colab CPU, but large enough to follow simple instructions.

In [ ]:
# Install transformers if needed (uncomment on Colab)
# !pip install transformers

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np

# Load SmolLM2-135M-Instruct (135M parameters --- loads in seconds, runs on CPU)
# attn_implementation="eager" lets us inspect attention weights later
print("Loading SmolLM2 model...")
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="eager")
model.eval()  # Set to evaluation mode
print(f"Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 1. How the Model Sees Text: Tokens

Before generating text, let's see how the model breaks text into pieces called **tokens**. Tokens are usually common words or word fragments.

In [ ]:
# Tokenize a sentence
sentence = "The quick brown fox jumps over the lazy dog"
tokens = tokenizer.encode(sentence)
print(f"Sentence: '{sentence}'")
print(f"Token IDs: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print()

# Decode each token to see what it represents
print("Token breakdown:")
for token_id in tokens:
    token_text = tokenizer.decode([token_id])
    print(f"  ID {token_id:5d} -> '{token_text}'")

In [ ]:
# Try a longer sentence
sentence2 = "Artificial intelligence is transforming how we live and work"
tokens2 = tokenizer.encode(sentence2)
print(f"Sentence: '{sentence2}'")
print(f"Number of tokens: {len(tokens2)}")
print()
print("Token breakdown:")
for token_id in tokens2:
    token_text = tokenizer.decode([token_id])
    print(f"  ID {token_id:5d} -> '{token_text}'")

Notice that common words like "the" and "is" are single tokens, but less common words might be split into pieces. SmolLM2's vocabulary has about 49,000 tokens.

## 2. Predicting the Next Word

The core of an LLM: given some text, predict what word comes next. Let's see what GPT-2 thinks is likely after different prompts.

In [ ]:
def predict_next_words(prompt, top_k=10):
    "Show the top-k most likely next words for a prompt."
    # Encode the prompt (returns both input_ids and attention_mask)
    inputs = tokenizer(prompt, return_tensors='pt')
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
        next_token_logits = outputs.logits[0, -1, :]  # Logits for the next token
    
    # Convert to probabilities
    probs = torch.softmax(next_token_logits, dim=0)
    
    # Get top-k predictions
    top_probs, top_indices = torch.topk(probs, top_k)
    
    print(f"Prompt: '{prompt}'")
    print(f"Top {top_k} predictions for the next word:")
    for i in range(top_k):
        token = tokenizer.decode([top_indices[i]])
        prob = top_probs[i].item()
        bar_len = int(prob * 50) if prob == prob else 0  # NaN guard
        bar = '#' * max(0, bar_len)
        print(f"  '{token}' ({prob*100:.1f}%) {bar}")
    print()

# Try different prompts
predict_next_words("The capital of France is")
predict_next_words("Once upon a")
predict_next_words("The meaning of life is")

Compare this to our Markov model from L18:
- The Markov model only looked at the **previous word**
- The LLM looks at **all previous words** and uses attention to figure out which ones matter

That's why "The capital of France is" → "Paris" works: the model attends to "capital" and "France" even though they're several words back.

## 3. Inside the Model: How Attention Works

So far we've treated the LLM as a black box. Let's open it up and watch the **attention mechanism** at work --- the same idea you saw in the slides.

The attention mechanism gives each word a new, **context-dependent** representation in three steps:

1. Compute a **score** between this word and every other word (dot product of their vectors)
2. Turn scores into **weights** with softmax
3. Replace the word's vector with a **weighted average** of all word vectors

That's it --- three lines of math. Modern transformers stack this hundreds of times across many "heads," but the core operation is what we'll build below.

In [ ]:
# Four toy word vectors (2D for easy visualization)
# Same example as the slides: "The spring season blooms"
words = ['the', 'spring', 'season', 'blooms']
vecs = np.array([
    [1.0, 0.0],   # the
    [2.0, 1.0],   # spring
    [2.0, 3.0],   # season
    [0.0, 2.0],   # blooms
])

# Compute attention from 'spring' (index 1) to every other word
focus = 1
scores = vecs @ vecs[focus]            # step 1: dot products
scores[focus] = -np.inf                # exclude self (matches the slide)

# Softmax to get attention weights
exp_scores = np.exp(scores - scores.max())
weights = exp_scores / exp_scores.sum()  # step 2: softmax

print(f"Attention weights from '{words[focus]}':")
for w, weight in zip(words, weights):
    bar = '#' * int(weight * 30)
    print(f"  {w:8s} ({weight:.2f}) {bar}")

# New context-aware representation = weighted average of all vectors
new_vec = (weights[:, None] * vecs).sum(axis=0)  # step 3: weighted sum
print(f"\nOriginal '{words[focus]}' vector:    {vecs[focus]}")
print(f"New context-aware vector:    {new_vec.round(2)}")

The new "spring" vector is pulled toward "season" (which got most of the attention) --- because in this sentence, "spring" *means* a time of year. In a different sentence ("The spring couch was soft"), "spring" would attend to "couch" instead and get a very different vector.

That's how attention solves the L19 limitation: **the same word gets different vectors depending on context**.

### Attention inside SmolLM2

Now let's see what attention SmolLM2 actually computes on a tricky pronoun-resolution sentence:

> "The animal didn't cross the street because it was too tired."

A human reads "it" and figures out it refers to **animal** (the animal got tired, not the street). Does the model do something similar?

Each layer of the transformer has multiple attention "heads," each learning a different pattern. After looking through SmolLM2's 30 layers × 9 heads, **layer 13, head 8** has clearly learned a pronoun-noun resolution pattern. Let's look at it.

In [ ]:
import matplotlib.pyplot as plt

sentence = "The animal didn't cross the street because it was too tired"
inputs = tokenizer(sentence, return_tensors='pt')
tokens = [tokenizer.decode([t]).strip() for t in inputs['input_ids'][0]]

# Run the model and ask for attention weights
with torch.no_grad():
    out = model(**inputs, output_attentions=True)

# out.attentions is a tuple of (n_layers,) each of shape (batch, heads, seq, seq)
# Pick layer 13, head 8 (specialized in pronoun -> noun)
attn = out.attentions[13][0, 8].numpy()  # shape (seq, seq)

# Show attention FROM 'it' to every other token
it_idx = tokens.index('it')
print(f"Attention from 'it' (layer 13, head 8):")
for i, t in enumerate(tokens):
    score = attn[it_idx, i]
    bar = '#' * int(score * 50)
    print(f"  [{i:2d}] {t!r:12s} ({score:.2f}) {bar}")

# Heatmap of the full attention matrix for this head
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn, cmap='Blues')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right')
ax.set_yticklabels(tokens)
ax.set_xlabel("attending TO")
ax.set_ylabel("attending FROM")
ax.set_title("Attention weights (layer 13, head 8)")
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

That single head has learned: **when looking at "it", attend to the closest preceding noun.** It's not perfect --- different heads do different things --- but you can see real linguistic structure emerging from training.

The full model has **30 layers × 9 heads = 270 different attention patterns**. Some specialize in syntax, some in semantics, some in copying tokens. Together they let the model handle context that a Markov model can't even represent.

## 4. Generating Text

Now let's generate longer text by repeatedly predicting the next word --- the same process as our Markov model, but with a much more powerful predictor.

In [ ]:
def generate_text(prompt, max_length=50, temperature=1.0):
    "Generate text from a prompt by sampling next tokens."
    inputs = tokenizer(prompt, return_tensors='pt')
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated

# Generate from different prompts
prompts = [
    "Once upon a time",
    "The best way to learn machine learning is",
    "In the year 2050, robots will",
]

for prompt in prompts:
    print(f"Prompt: '{prompt}'")
    result = generate_text(prompt, max_length=60)
    print(f"Generated: {result}")
    print()

## 5. Temperature: Controlling Creativity

Remember from the slides: **temperature** controls how "creative" the model is.

- **Low temperature** (0.3): picks the most likely words -> predictable, repetitive
- **High temperature** (1.5): gives rare words a chance -> creative, sometimes weird

In [ ]:
prompt = "The secret to happiness is"
print(f"Prompt: '{prompt}'")
print()

for temp in [0.3, 0.7, 1.0, 1.5]:
    print(f"Temperature = {temp}:")
    result = generate_text(prompt, max_length=50, temperature=temp)
    generated_part = result[len(prompt):]
    print(f"  ...{generated_part}")
    print()

Notice:
- At temperature 0.3, the model is very conservative and may repeat itself
- At temperature 1.0 (default), it's balanced between coherence and variety
- At temperature 1.5, it takes more risks --- sometimes creative, sometimes nonsensical

This is a direct consequence of how sampling from a probability distribution works: temperature reshapes the distribution to be more peaked (low) or more uniform (high).

## 6. Same Prompt, Different Outputs

Since the model **samples** from a probability distribution, running the same prompt twice gives different results. This is just like our Markov model from L18 --- randomness is built in.

In [ ]:
prompt = "A robot walks into a bar and"
print(f"Prompt: '{prompt}'")
print()

for i in range(5):
    result = generate_text(prompt, max_length=40, temperature=0.9)
    generated_part = result[len(prompt):]
    print(f"  Run {i+1}: ...{generated_part}")
    print()

Each run produces different text because the model samples randomly at each step. This is a feature, not a bug --- it's what makes LLMs creative rather than deterministic.

## 7. From Continuation to Conversation: Instruction Tuning

So far, the model has been doing one thing: **continuing whatever text we give it**. But SmolLM2 is also **instruction-tuned** --- after pre-training, it was fine-tuned on examples of `(question, answer)` pairs so it learns to *respond* rather than *continue*.

To trigger this behavior, we wrap our question in a special **chat template** that the model recognizes:

```
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
```

The model then generates the assistant's reply. The `apply_chat_template` helper formats this for us automatically.

In [ ]:
def chat(question, max_new_tokens=80, temperature=0.7):
    "Ask the instruction-tuned model a question and return its reply."
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (skip the prompt)
    answer = tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return answer.strip()

questions = [
    "What is the capital of France?",
    "List three primary colors.",
    "Why is the sky blue? Answer in one short sentence.",
    "Write a one-line haiku about machine learning.",
]
for q in questions:
    print(f"Q: {q}")
    print(f"A: {chat(q)}")
    print()

Notice the qualitative shift: with the chat template, the model **answers** rather than continuing. This is what makes ChatGPT-style assistants feel like assistants --- the *base* model is still just predicting the next token, but instruction tuning + chat templates teach it to follow user intent.

⚠️ **Reality check.** SmolLM2-135M is one of the smallest instruction-tuned LLMs. Its answers are short and sometimes wrong --- ask "What is 17 × 23?" and it may confidently give nonsense. Bigger models (Llama 3.1, GPT-4, etc.) follow the same recipe, just at much larger scale and with much more training data.

## 8. Connecting Back: From Markov to a Small LLM

Let's compare our Markov model (L18) and SmolLM2 on the same prompt.

In [ ]:
import nltk
nltk.download('gutenberg', quiet=True)

# Build a simple Markov model from Emma (from L18)
emma_words = [w.lower() for w in nltk.corpus.gutenberg.words('austen-emma.txt')]
markov_model = {}
for i in range(len(emma_words) - 1):
    curr = emma_words[i]
    nxt = emma_words[i + 1]
    if curr not in markov_model:
        markov_model[curr] = {}
    if nxt not in markov_model[curr]:
        markov_model[curr][nxt] = 0
    markov_model[curr][nxt] = markov_model[curr][nxt] + 1

import random
random.seed(42)

def markov_generate(start_word, n_words):
    result = [start_word]
    current = start_word
    for i in range(n_words - 1):
        if current not in markov_model:
            break
        candidates = markov_model[current]
        words = list(candidates.keys())
        counts = list(candidates.values())
        total = sum(counts)
        r = random.random()
        cumulative = 0
        for j in range(len(words)):
            cumulative = cumulative + counts[j] / total
            if r < cumulative:
                result.append(words[j])
                current = words[j]
                break
    return " ".join(result)

# Compare
print("=== Markov model (order 1, trained on Emma) ===")
for i in range(3):
    print(f"  {markov_generate('she', 20)}")
    
print()
print("=== SmolLM2-135M (trained on internet-scale text) ===")
for i in range(3):
    result = generate_text("She", max_length=25, temperature=0.8)
    print(f"  {result}")

The difference is dramatic:
- **Markov**: captures local word patterns but quickly loses coherence
- **SmolLM2**: maintains meaning, grammar, and topic across the full sentence --- and it has only 135M parameters! State-of-the-art models (Llama 3, GPT-4) use the same recipe at 1000× the scale.

Both use the same principle (predict the next word), but SmolLM2 has attention to look at all previous words, plus 135 million parameters trained on trillions of tokens.

## Summary

1. **Attention** lets the model decide which words to focus on when interpreting each word — three lines of math: score → softmax → weighted sum
2. **Transformers** stack many attention layers × many heads → increasingly rich, context-aware representations
3. **LLMs** = very large transformers that compute P(next word | all previous words)
4. **Generation** = repeatedly sampling the next word (same idea as Markov, but with full context)
5. **Temperature** controls creativity: low = safe, high = adventurous
6. **Instruction tuning** + **chat templates** turn a next-word predictor into an assistant that *answers* rather than *continues*
7. LLMs are powerful **pattern matchers** trained on massive data --- not reasoning engines
8. **Next (L21)**: ethics, biases, hallucinations, and how to use LLMs responsibly